In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/models/sanjayashrestha123/final-adapter/transformers/default/1/adapter_model.safetensors
/kaggle/input/models/sanjayashrestha123/final-adapter/transformers/default/1/runner.md
/kaggle/input/models/sanjayashrestha123/final-adapter/transformers/default/1/adapter_config.json
/kaggle/input/models/sanjayashrestha123/final-adapter/transformers/default/1/tokenizer.json
/kaggle/input/models/sanjayashrestha123/final-adapter/transformers/default/1/tokenizer_config.json
/kaggle/input/models/sanjayashrestha123/final-adapter/transformers/default/1/chat_template.jinja


In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("huggingface_token")
login(token=secret_value_0)

base_model_id = "meta-llama/Llama-2-7b-chat-hf"  # your base model
adapter_path = "path/to/adapters"         # folder with adapter_model.safetensors

# Load base model in full precision for merging
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Load and merge adapter
model = PeftModel.from_pretrained(model, adapter_path)
model = model.merge_and_unload()  # merges weights and removes adapter structure

# Save merged model
model.save_pretrained("/kaggle/working/")
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
tokenizer.save_pretrained("/kaggle/working/")